## Step 1: Install Dependencies

In [2]:
# import packages
import pandas as pd
import numpy as np
import matplotlib
import comtradeapicall as comtrade
import wbgapi as wb
import networkx
import sqlalchemy

## Step 2: Build Country Nodes from World Bank API

### Fetch GDP (current USD) and population for all countries, latest year

In [3]:
indicators = {"NY.GDP.MKTP.CD": "gdp_usd", "SP.POP.TOTL": "population"}
df_wb = wb.data.DataFrame(list(indicators.keys()), time=2022, labels=True)
df_wb = df_wb.rename(columns=indicators).reset_index()
df_wb.columns = ["country_code", "country_name", "gdp_usd", "population"]

In [4]:
df_wb.shape

(266, 4)

In [5]:
df_wb.columns

Index(['country_code', 'country_name', 'gdp_usd', 'population'], dtype='object')

### Get ISO3 + region metadata

In [6]:
meta = wb.economy.DataFrame()[["name", "region", "incomeLevel"]]
meta.index.name = "country_code"
meta = meta.reset_index()

In [7]:
nodes_df = df_wb.merge(meta, on="country_code", how="left")
nodes_df.to_csv("nodes.csv", index=False)
print(nodes_df.head())

  country_code           country_name       gdp_usd  population  \
0          ZWE               Zimbabwe  4.075756e+10  16069056.0   
1          ZMB                 Zambia  2.916378e+10  20152938.0   
2          YEM            Yemen, Rep.           NaN  38222876.0   
3          PSE     West Bank and Gaza  1.916550e+10   5043612.0   
4          VIR  Virgin Islands (U.S.)  4.672000e+09    105413.0   

                    name region incomeLevel  
0               Zimbabwe    SSF         LMC  
1                 Zambia    SSF         LMC  
2            Yemen, Rep.    MEA         LIC  
3     West Bank and Gaza    MEA         LMC  
4  Virgin Islands (U.S.)    LCN         HIC  


In [8]:
nodes_df.shape

(266, 7)

## Step 3: Fetch Bilateral Trade Edges via UN Comtrade

### Fetch total merchandise exports between all reporters and partners, 2022

In [9]:
trade_raw = comtrade.getFinalData(
    subscription_key="YOUR_API_KEY",   # free key from comtradedeveloper.un.org
    typeCode="C",        # commodities
    freqCode="A",        # annual
    clCode="HS",
    period="2022",
    reporterCode=None,   # all reporters
    cmdCode="TOTAL",     # total trade (no HS breakdown)
    flowCode="X",        # exports only; reporter -> partner
    partnerCode=None,
    partner2Code="0",
    customsCode="C00",
    motCode="0"
)

{ "statusCode": 401, "message": "Access denied due to invalid subscription key. Make sure to provide a valid key for an active subscription." }


In [10]:
# Build edge list: source=reporterISO, target=partnerISO, weight=trade_value_usd
trade_edges = trade_raw[["reporterISO", "partnerISO", "primaryValue"]].copy()
trade_edges.columns = ["source", "target", "trade_value_usd"]
trade_edges["edge_type"] = "trade"
trade_edges = trade_edges[trade_edges["target"] != "W00"]  # drop "World" aggregate
trade_edges.to_csv("trade_edges.csv", index=False)

TypeError: 'NoneType' object is not subscriptable

The `NoneType` error means `getFinalData()` returned `None` — the API call silently failed. This is a very common issue with the UN Comtrade API.